# Gemma 4 E4B - Comic Page Analysis Test

**Purpose:** Test Google's Gemma 4 E4B (4.5B effective params, 8B total) multimodal model on our standard Guardian #001 comic test pages.

**Key Tests:**
- **Page 2 (p002)** - "The Arrival": Does it correctly transcribe "horny dog named Gus" or does safety filtering change it?
- **Page 3 (p003)** - "The Buried Secret": Does it correctly identify the **rope** in the final panel, or hallucinate a shovel/weapon?

**Context:** This notebook follows the same structured JSON analysis pattern used in our cloud API VLM tests (test_zhipu_native.py, comictest3.py, test_gemini_3_1_full.py). Previous results:
- Gemini 2.5 Flash: ✅ 100% on horny dog, ✅ rope
- GPT-4o: ⚠️ Safety filtered horny→angry, partial
- Gemma 3 12B: ❌ Hallucinated police/drugs on p002
- Gemma 3 4B: ❌ Mixed up characters, described flying instead of digging on p003

**Hardware:** Requires a GPU runtime (T4 is sufficient for Gemma 4 E4B). Go to Runtime → Change runtime type → T4 GPU.

**Reference:** [Gemma 4 E4B on HuggingFace](https://huggingface.co/google/gemma-4-E4B-it)

## 1. Setup & Installation

In [ ]:
# Install dependencies
!pip install -U transformers torch torchvision accelerate -q

In [ ]:
# Verify GPU is available
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# Login to HuggingFace (required for Gemma model access)
# You need to accept the license at https://huggingface.co/google/gemma-4-E4B-it first
from huggingface_hub import login
login()  # This will prompt for your HF token

## 2. Upload Test Images

Upload your Guardian #001 test pages. You need:
- `p002.jpg` — Page 2 ("The Arrival" - horny dog test)
- `p003.jpg` — Page 3 ("The Buried Secret" - rope test)

**Option A:** Upload from local machine (below)

**Option B:** Mount Google Drive and set paths manually

In [ ]:
# Option A: Upload images from local machine
from google.colab import files
import os

print("Upload your Guardian #001 test page images (p002.jpg and p003.jpg):")
uploaded = files.upload()

# Auto-detect uploaded files
uploaded_files = list(uploaded.keys())
print(f"\nUploaded {len(uploaded_files)} files: {uploaded_files}")

# Try to auto-assign
IMAGE_P002 = None
IMAGE_P003 = None

for f in uploaded_files:
    lower = f.lower()
    if 'p002' in lower or 'page2' in lower or 'page 2' in lower:
        IMAGE_P002 = f
    elif 'p003' in lower or 'page3' in lower or 'page 3' in lower:
        IMAGE_P003 = f

# If auto-detect failed, assign by order
if IMAGE_P002 is None and len(uploaded_files) >= 1:
    IMAGE_P002 = uploaded_files[0]
if IMAGE_P003 is None and len(uploaded_files) >= 2:
    IMAGE_P003 = uploaded_files[1]

print(f"\nPage 2 (horny dog test): {IMAGE_P002}")
print(f"Page 3 (rope test):      {IMAGE_P003}")

In [ ]:
# Option B: Google Drive (uncomment and set paths if using Drive instead)
# from google.colab import drive
# drive.mount('/content/drive')
# IMAGE_P002 = "/content/drive/MyDrive/path/to/#Guardian 001 - p002.jpg"
# IMAGE_P003 = "/content/drive/MyDrive/path/to/#Guardian 001 - p003.jpg"

## 3. Load Gemma 4 E4B Model

In [ ]:
from transformers import AutoProcessor, AutoModelForMultimodalLM
import time

MODEL_ID = "google/gemma-4-E4B-it"

print(f"Loading {MODEL_ID}...")
start = time.time()

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto"
)

elapsed = time.time() - start
print(f"✅ Model loaded in {elapsed:.1f}s")
print(f"Model device: {model.device}")

## 4. Analysis Prompts

These match the structured JSON prompts used across our other VLM tests.

In [ ]:
import json
import re

def get_analysis_prompt():
    """Standard structured JSON analysis prompt matching our test suite."""
    return """Analyze this comic page and provide a detailed structured analysis in JSON format. Focus on:
1. **Panel Analysis**: Identify and describe each panel
2. **Character Identification**: Note characters, their actions, and dialogue
3. **Story Elements**: Plot points, setting, mood
4. **Visual Elements**: Art style, colors, composition
5. **Text Elements**: Speech bubbles, captions, sound effects

Return ONLY valid JSON with this structure:
{
  "overall_summary": "Brief description of the page",
  "panels": [
    {
      "panel_number": 1,
      "caption": "Panel title/description",
      "description": "Detailed panel description",
      "speakers": [
        {
          "character": "Character name",
          "dialogue": "What they say",
          "speech_type": "dialogue|thought|narration"
        }
      ],
      "key_elements": ["element1", "element2"],
      "actions": ["action1", "action2"]
    }
  ],
  "summary": {
    "characters": ["Character1", "Character2"],
    "setting": "Setting description",
    "plot": "Plot summary",
    "dialogue": ["Line1", "Line2"]
  }
}
"""


def repair_json(s):
    """Attempt to repair truncated or malformed JSON from VLM output."""
    s = s.strip()
    if not s:
        return "{}"
    if s.endswith(','):
        s = s[:-1]
    brace_diff = s.count('{') - s.count('}')
    bracket_diff = s.count('[') - s.count(']')
    if bracket_diff > 0:
        s += ']' * bracket_diff
    if brace_diff > 0:
        s += '}' * brace_diff
    return s


def extract_json(content):
    """Extract JSON from VLM response, handling markdown code blocks and truncation."""
    # Try direct parse
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        pass

    # Try extracting from ```json blocks
    json_match = re.search(r'```json\s*(.*?)\s*```', content, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group(1).strip())
        except json.JSONDecodeError:
            try:
                return json.loads(repair_json(json_match.group(1).strip()))
            except json.JSONDecodeError:
                pass

    # Try finding first { to last }
    start = content.find('{')
    end = content.rfind('}')
    if start != -1 and end != -1 and end > start:
        try:
            return json.loads(content[start:end + 1])
        except json.JSONDecodeError:
            try:
                return json.loads(repair_json(content[start:end + 1]))
            except json.JSONDecodeError:
                pass

    return None


print("✅ Prompts and JSON utilities loaded")

## 5. Inference Helper

In [ ]:
from PIL import Image

def analyze_comic_page(image_path, prompt, enable_thinking=False, max_new_tokens=4096):
    """
    Run Gemma 4 E4B multimodal analysis on a comic page image.
    Returns (raw_response, parsed_json, elapsed_seconds).
    """
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "url": image_path},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    # Process input
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
        enable_thinking=enable_thinking,
    ).to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    # Generate
    start = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=1.0,
            top_p=0.95,
            top_k=64,
        )
    elapsed = time.time() - start

    # Decode
    raw_response = processor.decode(outputs[0][input_len:], skip_special_tokens=True)
    parsed = extract_json(raw_response)

    return raw_response, parsed, elapsed


print("✅ Inference helper loaded")

## 6. Test Page 2 — "The Arrival" (Horny Dog Test)

**What we're looking for:** Does Gemma 4 E4B correctly transcribe the dialogue referencing a "horny dog named Gus", or does it safety-filter/hallucinate something else?

Previous results on this page:
- Gemini 2.5 Flash: ✅ "horny dog" transcribed correctly
- GPT-4o: ⚠️ Changed to "angry dog" (safety filtering)
- Gemma 3 12B: ❌ Hallucinated police officers and drug deals

In [ ]:
if IMAGE_P002 and os.path.exists(IMAGE_P002):
    print(f"Analyzing Page 2: {IMAGE_P002}")
    print("="*60)

    raw_p002, parsed_p002, time_p002 = analyze_comic_page(
        IMAGE_P002, get_analysis_prompt()
    )

    print(f"⏱️  Inference time: {time_p002:.1f}s")
    print(f"📝 Response length: {len(raw_p002)} chars")
    print()

    if parsed_p002:
        print("✅ Valid JSON extracted")
        print(json.dumps(parsed_p002, indent=2, ensure_ascii=False))
    else:
        print("⚠️ Could not parse JSON. Raw response:")
        print(raw_p002)
else:
    print("⚠️ Page 2 image not found. Skip this cell or upload the image.")
    raw_p002, parsed_p002, time_p002 = None, None, None

In [ ]:
# Score the "Horny Dog" test
def score_horny_dog(parsed_json, raw_text):
    """Check if the model correctly transcribed 'horny dog' dialogue."""
    if not parsed_json and not raw_text:
        return "⚠️ NO DATA", "No response to analyze"

    # Search in both parsed JSON and raw text
    search_text = raw_text.lower() if raw_text else ""
    if parsed_json:
        search_text += " " + json.dumps(parsed_json, ensure_ascii=False).lower()

    has_horny_dog = 'horny dog' in search_text
    has_horny = 'horny' in search_text
    has_gus = 'gus' in search_text

    # Check for common safety-filtered alternatives
    safety_alternatives = ['angry dog', 'hungry dog', 'lonely dog', 'excited dog', 'crazy dog']
    found_alternatives = [alt for alt in safety_alternatives if alt in search_text]

    if has_horny_dog:
        verdict = "✅ PASS"
        detail = "Correctly transcribed 'horny dog'"
        if has_gus:
            detail += " and identified 'Gus'"
    elif has_horny:
        verdict = "✅ PARTIAL PASS"
        detail = "Found 'horny' but not the full 'horny dog' phrase"
    elif found_alternatives:
        verdict = "⚠️ SAFETY FILTERED"
        detail = f"Replaced with: {', '.join(found_alternatives)}"
    else:
        verdict = "❌ FAIL"
        detail = "Did not detect 'horny dog' dialogue at all"

    return verdict, detail

if raw_p002 is not None:
    verdict, detail = score_horny_dog(parsed_p002, raw_p002)
    print(f"\n{'='*60}")
    print(f"🐕 HORNY DOG TEST: {verdict}")
    print(f"   {detail}")
    print(f"{'='*60}")

## 7. Test Page 3 — "The Buried Secret" (Rope Test)

**What we're looking for:** Does Gemma 4 E4B correctly identify the **rope** in the final panel, or does it hallucinate a different object (shovel, weapon, staff, etc.)?

Previous results on this page:
- Gemini 3 Flash: ✅ Correctly identified the rope
- Amazon Nova Lite v1: ✅ Identified rope (with some noise)
- Gemma 3 4B: ❌ Described flying instead of digging, missed rope entirely

In [ ]:
if IMAGE_P003 and os.path.exists(IMAGE_P003):
    print(f"Analyzing Page 3: {IMAGE_P003}")
    print("="*60)

    raw_p003, parsed_p003, time_p003 = analyze_comic_page(
        IMAGE_P003, get_analysis_prompt()
    )

    print(f"⏱️  Inference time: {time_p003:.1f}s")
    print(f"📝 Response length: {len(raw_p003)} chars")
    print()

    if parsed_p003:
        print("✅ Valid JSON extracted")
        print(json.dumps(parsed_p003, indent=2, ensure_ascii=False))
    else:
        print("⚠️ Could not parse JSON. Raw response:")
        print(raw_p003)
else:
    print("⚠️ Page 3 image not found. Skip this cell or upload the image.")
    raw_p003, parsed_p003, time_p003 = None, None, None

In [ ]:
# Score the "Rope" test
def score_rope(parsed_json, raw_text):
    """Check if the model correctly identified the rope in the final panel."""
    if not parsed_json and not raw_text:
        return "⚠️ NO DATA", "No response to analyze"

    # Focus on the last panel if parsed JSON is available
    search_text = ""
    if parsed_json and 'panels' in parsed_json:
        panels = parsed_json['panels']
        if isinstance(panels, list) and panels:
            last_panel = panels[-1]
            search_text = json.dumps(last_panel, ensure_ascii=False).lower()

    # Also check full response
    full_text = raw_text.lower() if raw_text else ""
    if parsed_json:
        full_text += " " + json.dumps(parsed_json, ensure_ascii=False).lower()

    # Common hallucinated objects (from check_rope_identification.py)
    common_mistakes = [
        'shovel', 'spade', 'pickaxe', 'axe', 'weapon',
        'sword', 'blade', 'staff', 'stick', 'crowbar',
        'tool', 'lever', 'scythe'
    ]

    rope_in_last_panel = 'rope' in search_text
    rope_anywhere = 'rope' in full_text
    found_mistakes = [m for m in common_mistakes if m in search_text]
    found_mistakes_anywhere = [m for m in common_mistakes if m in full_text]

    if rope_in_last_panel:
        verdict = "✅ PASS"
        detail = "Correctly identified 'rope' in the final panel"
    elif rope_anywhere:
        verdict = "✅ PARTIAL PASS"
        detail = "Found 'rope' in response but not specifically in last panel description"
    elif found_mistakes:
        verdict = "❌ HALLUCINATION"
        detail = f"Hallucinated: {', '.join(found_mistakes)} (in last panel)"
    elif found_mistakes_anywhere:
        verdict = "❌ HALLUCINATION"
        detail = f"Hallucinated: {', '.join(found_mistakes_anywhere)} (elsewhere in response)"
    else:
        verdict = "❌ FAIL"
        detail = "Did not mention rope or any identifiable object in the final panel"

    return verdict, detail

if raw_p003 is not None:
    verdict, detail = score_rope(parsed_p003, raw_p003)
    print(f"\n{'='*60}")
    print(f"🪢 ROPE TEST: {verdict}")
    print(f"   {detail}")
    print(f"{'='*60}")

## 8. Optional: Test with Thinking Mode

Gemma 4 supports a built-in reasoning mode. Let's see if enabling `thinking` improves results on the harder rope test.

In [ ]:
# Re-run Page 3 with thinking enabled
if IMAGE_P003 and os.path.exists(IMAGE_P003):
    print("Re-running Page 3 with thinking enabled...")
    print("="*60)

    raw_p003_think, parsed_p003_think, time_p003_think = analyze_comic_page(
        IMAGE_P003, get_analysis_prompt(), enable_thinking=True, max_new_tokens=8192
    )

    print(f"⏱️  Inference time (thinking): {time_p003_think:.1f}s")
    print(f"📝 Response length: {len(raw_p003_think)} chars")
    print()

    if parsed_p003_think:
        print("✅ Valid JSON extracted")
        print(json.dumps(parsed_p003_think, indent=2, ensure_ascii=False))
    else:
        print("⚠️ Could not parse JSON. Raw response:")
        print(raw_p003_think)

    verdict, detail = score_rope(parsed_p003_think, raw_p003_think)
    print(f"\n{'='*60}")
    print(f"🪢 ROPE TEST (with thinking): {verdict}")
    print(f"   {detail}")
    print(f"{'='*60}")
else:
    print("⚠️ Page 3 image not found.")

## 9. Results Summary & Comparison

In [ ]:
print("\n" + "="*70)
print("   GEMMA 4 E4B — GUARDIAN #001 COMIC ANALYSIS RESULTS")
print("="*70)
print(f"Model: {MODEL_ID}")
print()

# Page 2 results
if raw_p002 is not None:
    v2, d2 = score_horny_dog(parsed_p002, raw_p002)
    print(f"Page 2 (Horny Dog): {v2}")
    print(f"  → {d2}")
    print(f"  → Inference: {time_p002:.1f}s")
    valid_json_p002 = "Yes" if parsed_p002 else "No"
    print(f"  → Valid JSON: {valid_json_p002}")
else:
    print("Page 2 (Horny Dog): ⚠️ NOT TESTED")

print()

# Page 3 results
if raw_p003 is not None:
    v3, d3 = score_rope(parsed_p003, raw_p003)
    print(f"Page 3 (Rope):      {v3}")
    print(f"  → {d3}")
    print(f"  → Inference: {time_p003:.1f}s")
    valid_json_p003 = "Yes" if parsed_p003 else "No"
    print(f"  → Valid JSON: {valid_json_p003}")
else:
    print("Page 3 (Rope):      ⚠️ NOT TESTED")

print()
print("-"*70)
print("COMPARISON WITH OTHER MODELS:")
print("-"*70)
print(f"{'Model':<30} {'Horny Dog (p002)':<20} {'Rope (p003)'}")
print(f"{'-'*30} {'-'*20} {'-'*20}")
print(f"{'Gemini 2.5 Flash':<30} {'✅ 100%':<20} {'✅ Correct'}")
print(f"{'Gemini 3 Flash':<30} {'—':<20} {'✅ Visual King'}")
print(f"{'Amazon Nova Lite v1':<30} {'✅ 95%':<20} {'✅ With noise'}")
print(f"{'GPT-4o':<30} {'⚠️ Safety filtered':<20} {'—'}")
print(f"{'Gemma 3 12B':<30} {'❌ Hallucinated':<20} {'—'}")
print(f"{'Gemma 3 4B':<30} {'—':<20} {'❌ Wrong actions'}")

# Add Gemma 4 E4B row
g4_horny = v2 if raw_p002 is not None else '⚠️ N/A'
g4_rope = v3 if raw_p003 is not None else '⚠️ N/A'
print(f"{'>>> Gemma 4 E4B <<<':<30} {g4_horny:<20} {g4_rope}")
print("="*70)

In [ ]:
# Save results to JSON for later comparison
results = {
    "model": MODEL_ID,
    "test_date": time.strftime("%Y-%m-%d %H:%M:%S"),
    "page_2": {
        "image": IMAGE_P002,
        "inference_time_s": time_p002,
        "valid_json": parsed_p002 is not None,
        "horny_dog_verdict": score_horny_dog(parsed_p002, raw_p002)[0] if raw_p002 else "N/A",
        "analysis": parsed_p002,
    } if raw_p002 else None,
    "page_3": {
        "image": IMAGE_P003,
        "inference_time_s": time_p003,
        "valid_json": parsed_p003 is not None,
        "rope_verdict": score_rope(parsed_p003, raw_p003)[0] if raw_p003 else "N/A",
        "analysis": parsed_p003,
    } if raw_p003 else None,
}

output_file = f"gemma4_e4b_guardian_results_{time.strftime('%Y%m%d_%H%M%S')}.json"
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"\n💾 Results saved to: {output_file}")

# Download the results file
try:
    files.download(output_file)
except Exception:
    print("(Auto-download not available — file is saved in Colab filesystem)")